<a href="https://colab.research.google.com/github/2403a52208-sudo/NLP/blob/main/2403A52208_B09_Assg12.2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install Gensim for Word2Vec

In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 40.3 MB/s eta 0:00:00


 Numerical operations, Keras modules, Gensim for Word2Vec

In [ ]:
import numpy as np

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Conv1D
from tensorflow.keras.layers import GlobalMaxPooling1D, Dense, Concatenate
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
from gensim.models import Word2Vec

STEP 1:Sample text data (sentences)

In [ ]:

texts = [
    "data science is interesting",
    "machine learning is powerful",
    "natural language processing is fun",
    "gensim is useful for topic modeling",
    "this movie is great",
    "i love this film",
   ]
# Labels: 1 = positive, 0 = negative
labels = [1,1,0,0,1,0]



STEP 2: TOKENIZATION

In [ ]:
tokenizer = Tokenizer()

# Learn vocabulary
tokenizer.fit_on_texts(texts)

# Convert sentences into sequences
sequences = tokenizer.texts_to_sequences(texts)

# Vocabulary dictionary
word_index = tokenizer.word_index

print(word_index)

{'is': 1, 'this': 2, 'data': 3, 'science': 4, 'interesting': 5, 'machine': 6, 'learning': 7, 'powerful': 8, 'natural': 9, 'language': 10, 'processing': 11, 'fun': 12, 'gensim': 13, 'useful': 14, 'for': 15, 'topic': 16, 'modeling': 17, 'movie': 18, 'great': 19, 'i': 20, 'love': 21, 'film': 22}


In [ ]:
print(sequences)

[[3, 4, 1, 5], [6, 7, 1, 8], [9, 10, 11, 1, 12], [13, 1, 14, 15, 16, 17], [2, 18, 1, 19], [20, 21, 2, 22]]


STEP 3: PADDING

In [ ]:
max_length = 6

X = pad_sequences(sequences, maxlen=max_length)

y = np.array(labels)

In [ ]:
X

array([[ 0,  0,  3,  4,  1,  5],
       [ 0,  0,  6,  7,  1,  8],
       [ 0,  9, 10, 11,  1, 12],
       [13,  1, 14, 15, 16, 17],
       [ 0,  0,  2, 18,  1, 19],
       [ 0,  0, 20, 21,  2, 22]], dtype=int32)

STEP 4: TRAIN WORD2VEC MODEL

In [ ]:
sentences = [text.split() for text in texts]

# Train Word2Vec
w2v_model = Word2Vec(
    sentences,
    vector_size=50,
    window=3,
    min_count=1
)

STEP 5: CREATE EMBEDDING MATRIX


In [ ]:

vocab_size = len(word_index) + 1
embedding_dim = 50

embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, index in word_index.items():
    if word in w2v_model.wv:
        embedding_matrix[index] = w2v_model.wv[word]

In [ ]:
embedding_matrix[1]

array([-1.07245450e-03,  4.72862710e-04,  1.02066994e-02,  1.80185456e-02,
       -1.86058991e-02, -1.42336180e-02,  1.29177449e-02,  1.79459769e-02,
       -1.00308564e-02, -7.52674323e-03,  1.47610093e-02, -3.06694279e-03,
       -9.07322671e-03,  1.31081035e-02, -9.72032081e-03, -3.63203534e-03,
        5.75315952e-03,  1.98374758e-03, -1.65704302e-02, -1.88976359e-02,
        1.46235321e-02,  1.01405242e-02,  1.35153867e-02,  1.52573106e-03,
        1.27017805e-02, -6.81073172e-03, -1.89280277e-03,  1.15371468e-02,
       -1.50432754e-02, -7.87220709e-03, -1.50231645e-02, -1.86008448e-03,
        1.90762375e-02, -1.46383336e-02, -4.66753729e-03, -3.87548213e-03,
        1.61548741e-02, -1.18617918e-02,  9.03248801e-05, -9.50746797e-03,
       -1.92071013e-02,  1.00145862e-02, -1.75191704e-02, -8.78365058e-03,
       -7.01999670e-05, -5.92362892e-04, -1.53224804e-02,  1.92294866e-02,
        9.96411592e-03,  1.84662864e-02])

STEP 6: BUILD CNN MODEL

In [ ]:
input_layer = Input(shape=(max_length,))

In [ ]:
embedding_layer = Embedding(
    input_dim=vocab_size,
    output_dim=embedding_dim,
    weights=[embedding_matrix],  # Word2Vec weights
    input_length=max_length,
    trainable=False               # keep embeddings fixed
)(input_layer)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
conv1 = Conv1D(filters=100, kernel_size=3, activation='relu')(embedding_layer)

conv2 = Conv1D(filters=100, kernel_size=4, activation='relu')(embedding_layer)

conv3 = Conv1D(filters=100, kernel_size=5, activation='relu')(embedding_layer)

In [ ]:
pool1 = GlobalMaxPooling1D()(conv1)
pool2 = GlobalMaxPooling1D()(conv2)
pool3 = GlobalMaxPooling1D()(conv3)

In [ ]:
merged = Concatenate()([pool1, pool2, pool3])
dense = Dense(10, activation='relu')(merged)
output = Dense(1, activation='sigmoid')(dense)
model = Model(inputs=input_layer, outputs=output)


In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 6)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 6, 50)     │      1,150 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 4, 100)    │     15,100 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, 3, 100)    │     20,100 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_5 (Conv1D)   │ (None, 2, 100)    │     25,100 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 100)       │          0 │ conv1d_3[0][0]    │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 100)       │          0 │ conv1d_4[0][0]    │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 100)       │          0 │ conv1d_5[0][0]    │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 300)       │          0 │ global_max_pooli… │
│ (Concatenate)       │                   │            │ global_max_pooli… │
│                     │                   │            │ global_max_pooli… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 10)        │      3,010 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 1)         │         11 │ dense[0][0]       │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 64,471 (251.84 KB)

 Trainable params: 63,321 (247.35 KB)

 Non-trainable params: 1,150 (4.49 KB)

STEP 7: TRAIN MODEL

In [ ]:

model.fit(
    X,
    y,
    epochs=10,
    batch_size=2
)

Epoch 1/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.5000 - loss: 0.6926
Epoch 2/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6667 - loss: 0.6831
Epoch 3/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.6667 - loss: 0.6730
Epoch 4/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6667 - loss: 0.6624
Epoch 5/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8333 - loss: 0.6510
Epoch 6/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8333 - loss: 0.6387
Epoch 7/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8333 - loss: 0.6274
Epoch 8/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8333 - loss: 0.6137
Epoch 9/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 1.0000 - loss: 0.5982
Epoch 10/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 1.0000 - loss: 0.5817


 STEP 8: TEST MODEL

In [ ]:
test_text = ["this movie is amazing"]

seq = tokenizer.texts_to_sequences(test_text)

pad = pad_sequences(seq, maxlen=max_length)

prediction = model.predict(pad)

print(prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 133ms/step
[[0.57096547]]
